In [ ]:
#----core librarie---------
import pandas as pd 
import numpy as np

#----visulisation-----------
import matplotlib.pyplot as plt
%matplotlib inline 
import seaborn as sns 

#---- machine learning-------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import kagglehub


## **1.load dataset**

In [ ]:
#LOAD THE DATASET
df=pd.read_csv("D:/yash - downloads/dataset/food_delivery_analytics_cleaned.csv")
# df=pd.read_csv('https://kaggle.com/input/datasets/deepeshkansotia/food-delivery-operations-and-customer-analytics/food_delivery_analytics_cleaned.csv')
print(f'dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')# shows row x columns 
df.head()

---
now we know there are 30 features 15000 delivery order(instances)and our target is `delayed_delivery_flag` As it tells 'True' = delayed,'False' = on time

---

## **2 . Explore and understand the data**

In [ ]:

print('=== Data Types & Non-Null Counts ===')
df.info()

#we can clearly see the features having non numeric datatype is order_id(object) and 6 features(bool).

In [ ]:
# bool_cols=df.select_dtypes(include='bool').columns.tolist()
# print(bool_cols) #it show all the name of columns having bool datatype

In [ ]:
# statistical report
print('=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
# checking how many delivery are delayed vs delivery on time 
target_count=df['delayed_delivery_flag'].value_counts() # counts true and false enteries 
target_percent=df['delayed_delivery_flag'].value_counts(normalize=True)*100

summary=pd.DataFrame({'count':target_count,'Percentage(%)':target_percent.round(1)}) 
summary.index=['on time (False)','Delayed (True)']
print("======== Target variable distribution=======")
print(summary)



In [ ]:
# Plot it
fig = plt.subplots(figsize=(6, 4))
ax=sns.countplot(x='delayed_delivery_flag',data=df,palette=['#2ecc71','#e74c3c'])
plt.title("delayed vs on time deliveries",fontsize=14,fontweight='bold')
plt.xlabel("Delivery status")
plt.ylabel('Number of orders')
plt.xticks(ticks=[0,1],labels=['on time','delayed'])

total=len(df)
for p in ax.patches:
    percentage=f'{100* p.get_height()/ total:.1f}%'
    ax.annotate(f'{int(p.get_height()):,}\n({percentage})', (p.get_x() + 0.3, p.get_height() + 50))

plt.tight_layout()
plt.show()


## **3.handle missing value**

In [ ]:
# handling missing value 
missing=df.isnull().sum()
missing=missing[missing > 0].sort_values(ascending=False) # missing>0 considers only those with some null count 
print('==== Columns with missing values =====')
print(missing)
print(f" Total missing cells: {df.isnull().sum().sum()}")#writing .sum() after isnull().sum() addes all the count 

In [ ]:
#Fill numeric columns with the Median
#(median is safer than mean for skewed data)
numeric_cols=df.select_dtypes(include='number').columns # it collect all the columns with numeric data type 
df[numeric_cols]=df[numeric_cols].fillna(df[numeric_cols].median())

# Fill boolean columns with the MODE (most common value)
bool_cols = df.select_dtypes(include='bool').columns
for col in bool_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f'✅ Missing values after cleaning: {df.isnull().sum().sum()}')



## **4.VISUALIZE KEY PATTERNS**
let's look at which features seem most related to delivery delays 

In [ ]:
fig, axes=plt.subplots(2,2, figsize=(14,10))
fig.suptitle('key factors affecting Delivery Delays',fontsize=16,fontweight='bold')
# 1. Traffic level vs Delay
sns.boxplot(
    data=df, x='delayed_delivery_flag', y='traffic_level_score',
    palette=['#2ecc71', '#e74c3c'], ax=axes[0, 0]
)
axes[0, 0].set_title('Traffic Level vs Delay')
axes[0, 0].set_xticklabels(['On Time', 'Delayed'])

# 2. Delivery distance vs Delay
sns.boxplot(
    data=df, x='delayed_delivery_flag', y='delivery_distance_km',
    palette=['#2ecc71', '#e74c3c'], ax=axes[0, 1]
)
axes[0, 1].set_title('Delivery Distance vs Delay')
axes[0, 1].set_xticklabels(['On Time', 'Delayed'])

# 3. Preparation time vs Delay
sns.boxplot(
    data=df, x='delayed_delivery_flag', y='preparation_time_minutes',
    palette=['#2ecc71', '#e74c3c'], ax=axes[1, 0]
)
axes[1, 0].set_title('Preparation Time vs Delay')
axes[1, 0].set_xticklabels(['On Time', 'Delayed'])

# 4. Delivery Efficiency Score vs Delay
sns.boxplot(
    data=df, x='delayed_delivery_flag', y='delivery_efficiency_score',
    palette=['#2ecc71', '#e74c3c'], ax=axes[1, 1]
)
axes[1, 1].set_title('Delivery Efficiency Score vs Delay')
axes[1, 1].set_xticklabels(['On Time', 'Delayed'])

plt.tight_layout()
plt.show()

In [ ]:
fig , ax = plt.subplots(figsize=(18,12))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr , dtype=bool)) # hide upper triangle 
sns.heatmap(
    corr , mask=mask,annot=True, fmt='.2f',
    cmap='RdYlGn',center=0 , linewidths=0.5 , ax=ax
)
ax.set_title('feature correlation matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## **5.FEATURE ENGINEERING**

We create new useful feature from existing ones, then prepare data for the model.

In [ ]:
#-New feature 1:was the delay gap large?
# How much did the actual delivery time differ from the estimate?
df['delivery_delay_gap']=df['delivery_time_minutes']-df['estimated_delivery_time']
# ── New Feature 2: Combined stress score ─────────────────────
# High traffic + bad weather = more stress on delivery
df['stress_score'] = df['traffic_level_score'] * df['weather_severity_score']

# ── New Feature 3: Is it a peak hour? ────────────────────────
# Lunch (11-14) and Dinner (18-22) are typically busy hours
df['is_peak_hour'] = df['order_hour'].between(11, 14) | df['order_hour'].between(18, 22)
df['is_peak_hour'] = df['is_peak_hour'].astype(int)

# ── New Feature 4: Speed of partner (km per minute) ──────────
df['delivery_speed'] = df['delivery_distance_km'] / (df['delivery_time_minutes'] + 1)

print('✅ 4 new features created!')
print('  → delivery_delay_gap')
print('  → stress_score')
print('  → is_peak_hour')
print('  → delivery_speed') 


In [ ]:
# Drop non-useful columns 
drop_cols = ['order_id'] # ID is not useful for prediction
df_model =df.drop(columns=drop_cols)

#convert boolen columns to integer(0 or 1)
#ML models work with numbers, not True/False
for col in df_model.select_dtypes(include='bool').columns:
    df_model[col]=df_model[col].astype(int)
# defining feature x and target y .
TARGET='delayed_delivery_flag'
X=df_model.drop(columns=[TARGET])
y=df_model[TARGET]

print(f'features (x): {X.shape[1]} columns')
print(f'target (y):{y.value_counts().to_dict()}')


In [ ]:
#_____train / test split________
# we will use 80% to train and 20% to test
# stratify=y makes sure delayed orders appear in both train & test
X_train,X_test,y_train,y_test=train_test_split(X, y, test_size=0.20, random_state=42,stratify=y )

# scale featres
# bring all features to same scale (important for randomforest logistic ...)
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train)
X_test_sc=scaler.transform(X_test)

print(f'Training set size : {X_train.shape[0]:,} samples')
print(f' Test    set size : {X_test.shape[0]:,} sample')

## **6.Build & train ml models** 

In [ ]:
from sklearn.model_selection import  cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)


In [ ]:
# ── Model 1: Logistic Regression ─────────────────────────────
# Think of it as drawing a straight line to separate classes
lr = LogisticRegression(
    class_weight='balanced',   # handles the imbalance (only 9.5% delayed)
    max_iter=1000,
    random_state=42
)
lr.fit(X_train_sc, y_train)
print('✅ Logistic Regression trained!')

# ── Model 2: Random Forest ────────────────────────────────────
# Builds many decision trees and combines their votes
rf = RandomForestClassifier(
    n_estimators=200,          # number of trees
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1                  # use all CPU cores
)
rf.fit(X_train, y_train)
print('✅ Random Forest trained!')

# ── Model 3: Gradient Boosting ────────────────────────────────
# Builds trees one by one, each fixing the errors of the previous
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=5,
    random_state=42
)
gb.fit(X_train, y_train)
print('✅ Gradient Boosting trained!')

## 7.**Evaluate the models**
We measure each model using:
- **Accuracy** — how many predictions are correct?
- **AUC-ROC** — how well can the model rank delayed vs on-time?
- **Precision / Recall** — especially important for the delayed class (minority)

In [ ]:
# Defining a function to use repatedly for model evaluation 
def evaluate_model(name , model, x_test_data,y_test_data):
    
    y_pred = model.predict(x_test_data)
    y_pred_prob=model.predict_proba(x_test_data)[:,1]# important to add [:,1] as it only takes 1d array

    acc = accuracy_score(y_test_data, y_pred)
    auc = roc_auc_score(y_test_data, y_pred_prob)

    print(f'\n{'='*50}')
    print(f'📌 {name}')
    print(f'{'='*50}')
    print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  AUC-ROC  : {auc:.4f}')
    print()
    print(classification_report(y_test_data, y_pred, target_names=['On Time', 'Delayed']))

    return auc, y_pred_prob

auc_lr, prob_lr = evaluate_model('Logistic Regression',  lr, X_test_sc, y_test)
auc_rf, prob_rf = evaluate_model('Random Forest',        rf, X_test,    y_test)
auc_gb, prob_gb = evaluate_model('Gradient Boosting',    gb, X_test,    y_test)

In [ ]:
#--feature importance-what drives delays?-----

importance=pd.DataFrame({'feature':X_train.columns,
                         'Importance':rf.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=importance, x='Importance', y='feature',
    palette='viridis', ax=ax
)
ax.set_title('Top 15 Features That Predict Delivery Delays', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix for Best Model (Random Forest) ──────────
best_pred = rf.predict(X_test)
cm = confusion_matrix(y_test, best_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['On Time', 'Delayed'],
    yticklabels=['On Time', 'Delayed'], ax=ax
)
ax.set_title('Confusion Matrix — Random Forest (Best Model)', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
plt.tight_layout()
plt.show()